
# NTO Stage 2 Team — BERT + LightGBM Ranker (NDCG@20)

Этот ноутбук реализует end‑to‑end решение для задачи ранжирования книг:

- трёхуровневая релевантность: прочитано (2), в планах (1), «холодный» кандидат (0);
- BERT‑эмбеддинги описаний книг (сжатые через SVD);
- агрегированные user / book / user–book фичи;
- LightGBM Ranker (LambdaMART) с группами по `user_id` под метрику NDCG@20;
- формирование submission в требуемом формате `user_id,book_id_list`.


In [1]:
import os
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD

import lightgbm as lgb
from tqdm import tqdm

import math
import json
import warnings
warnings.filterwarnings("ignore")

# BERT / Transformers
try:
    import transformers  # type: ignore
    from transformers import AutoTokenizer, AutoModel  # type: ignore
    import torch  # type: ignore
except ImportError:
    # Если библиотек нет в окружении — раскомментировать строку ниже и выполнить ячейку
    # !pip install -q transformers sentencepiece
    from transformers import AutoTokenizer, AutoModel  # type: ignore
    import torch  # type: ignore

print("Torch device:", "cuda" if torch.cuda.is_available() else "cpu")


Torch device: cuda


In [2]:
# ==========================
# Пути и константы
# ==========================

DATA_DIR = Path("/kaggle/input/nto-2-tour-team")          # сюда распаковать stage2_team_data
PROCESSED_DIR = Path("./data/processed")
MODEL_DIR = Path("./models")
SUBMISSION_DIR = Path("./submissions")

for d in [PROCESSED_DIR, MODEL_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Имена файлов (подправь при необходимости)
TRAIN_PATH = DATA_DIR / "train.csv"
TARGETS_PATH = DATA_DIR / "targets.csv"
CANDIDATES_PATH = DATA_DIR / "candidates.csv"
BOOKS_PATH = DATA_DIR / "books.csv"
USERS_PATH = DATA_DIR / "users.csv"
GENRES_PATH = DATA_DIR / "genres.csv"
BOOK_GENRES_PATH = DATA_DIR / "book_genres.csv"
BOOK_DESCR_PATH = DATA_DIR / "book_descriptions.csv"

# Колонки
COL_USER_ID = "user_id"
COL_BOOK_ID = "book_id"
COL_HAS_READ = "has_read"          # 1 — прочитал, 0 — в планах
COL_TIMESTAMP = "timestamp"        # datetime
COL_RATING = "rating"              # 0–10, для has_read=0 обычно 0
COL_REL = "rel_target"             # трёхуровневая релевантность

# Параметры обучения
RANDOM_STATE = 42
NDCG_K = 20
NEGATIVES_PER_USER = 15

# BERT
BERT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
BERT_EMB_DIM = 384      # размер эмбеддинга для этой модели
BERT_SVD_DIM = 64       # во сколько сжимаем описания
BERT_BATCH_SIZE = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Имена файлов модели / submission
MODEL_FILENAME = "lgbm_ranker_bert.txt"
SUBMISSION_FILENAME = "submission_bert_ltr.csv"


## 1. Загрузка данных

In [3]:
def load_data() -> tuple[
    pd.DataFrame, pd.DataFrame, pd.DataFrame,
    pd.DataFrame, pd.DataFrame, pd.DataFrame,
    pd.DataFrame, pd.DataFrame
]:
    """Загрузка всех необходимых таблиц.

    При необходимости можно поменять sep=';' для метаданных.
    """
    print("Загружаем данные...")

    train_df = pd.read_csv(TRAIN_PATH)  # timestamp — строка datetime
    targets_df = pd.read_csv(TARGETS_PATH)
    candidates_df = pd.read_csv(CANDIDATES_PATH)
    books_df = pd.read_csv(BOOKS_PATH)
    users_df = pd.read_csv(USERS_PATH)
    genres_df = pd.read_csv(GENRES_PATH)
    book_genres_df = pd.read_csv(BOOK_GENRES_PATH)
    book_descr_df = pd.read_csv(BOOK_DESCR_PATH)

    print("train:", train_df.shape)
    print("targets:", targets_df.shape)
    print("candidates:", candidates_df.shape)
    print("books:", books_df.shape)
    print("users:", users_df.shape)
    print("genres:", genres_df.shape)
    print("book_genres:", book_genres_df.shape)
    print("book_descriptions:", book_descr_df.shape)

    return (
        train_df,
        targets_df,
        candidates_df,
        books_df,
        users_df,
        genres_df,
        book_genres_df,
        book_descr_df,
    )

(
    train_df,
    targets_df,
    candidates_df,
    books_df,
    users_df,
    genres_df,
    book_genres_df,
    book_descr_df,
) = load_data()

train_df.head()


Загружаем данные...
train: (269061, 5)
targets: (3512, 1)
candidates: (3512, 2)
books: (55785, 8)
users: (7289, 3)
genres: (439, 3)
book_genres: (103646, 2)
book_descriptions: (55784, 2)


,user_id,book_id,has_read,rating,timestamp
0,3870,310170,0,0,2008-04-27 21:06:16
1,3870,306406,0,0,2008-06-07 11:51:01
2,4091,195676,0,0,2008-08-06 00:40:55
3,3870,554261,1,8,2008-08-07 09:16:12
4,3870,33078,1,2,2008-08-07 09:17:20


## 2. BERT‑эмбеддинги описаний книг + SVD

In [4]:
# Колонка с описанием
DESC_COL = "description"
book_descr_df[DESC_COL] = book_descr_df[DESC_COL].fillna("").astype(str)

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = AutoModel.from_pretrained(BERT_MODEL_NAME)
bert_model.to(DEVICE)
bert_model.eval()

def bert_encode_texts(texts: list[str], batch_size: int = 128) -> np.ndarray:
    """Возвращает массив [len(texts), hidden_size] — mean pooling по токенам."""
    all_embeddings: list[np.ndarray] = []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERT encoding"):
        batch_texts = texts[i : i + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = bert_model(**encoded)
            last_hidden = outputs.last_hidden_state  # [bs, seq_len, hidden]
            emb = last_hidden.mean(dim=1).cpu().numpy()
        all_embeddings.append(emb)
    return np.vstack(all_embeddings)

BERT_EMB_CACHE = PROCESSED_DIR / "book_bert_embeddings.npz"
BERT_SVD_CACHE = PROCESSED_DIR / "book_bert_svd_embeddings.parquet"

if BERT_SVD_CACHE.exists():
    print("Загружаем BERT+SVD эмбеддинги из кэша...")
    bert_svd_df = pd.read_parquet(BERT_SVD_CACHE)
else:
    if BERT_EMB_CACHE.exists():
        print("Загружаем сырые BERT‑эмбеддинги из кэша...")
        data = np.load(BERT_EMB_CACHE)
        book_ids_order = data["book_ids"]
        bert_embeddings = data["embeddings"]
    else:
        print("Считаем BERT‑эмбеддинги описаний книг...")
        book_descr_sorted = book_descr_df.sort_values(COL_BOOK_ID).reset_index(drop=True)
        texts = book_descr_sorted[DESC_COL].tolist()
        book_ids_order = book_descr_sorted[COL_BOOK_ID].values

        bert_embeddings = bert_encode_texts(texts, batch_size=BERT_BATCH_SIZE)
        np.savez_compressed(
            BERT_EMB_CACHE,
            book_ids=book_ids_order,
            embeddings=bert_embeddings,
        )
        print("Сырые BERT‑эмбеддинги сохранены в", BERT_EMB_CACHE)

    print(f"Сжимаем эмбеддинги до {BERT_SVD_DIM} компонент через TruncatedSVD...")
    svd = TruncatedSVD(
        n_components=BERT_SVD_DIM,
        random_state=RANDOM_STATE,
    )
    bert_svd = svd.fit_transform(bert_embeddings)

    bert_cols = [f"bert_svd_{i}" for i in range(BERT_SVD_DIM)]
    bert_svd_df = pd.DataFrame(bert_svd, columns=bert_cols)
    bert_svd_df[COL_BOOK_ID] = book_ids_order
    bert_svd_df.to_parquet(BERT_SVD_CACHE, index=False)
    print("BERT+SVD эмбеддинги сохранены в", BERT_SVD_CACHE)

bert_svd_df.head()


tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

2025-12-04 18:47:44.787678: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764874064.997077      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764874065.054218      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Считаем BERT‑эмбеддинги описаний книг...


BERT encoding: 100%|██████████| 436/436 [02:34<00:00,  2.83it/s]


Сырые BERT‑эмбеддинги сохранены в data/processed/book_bert_embeddings.npz
Сжимаем эмбеддинги до 64 компонент через TruncatedSVD...
BERT+SVD эмбеддинги сохранены в data/processed/book_bert_svd_embeddings.parquet


,bert_svd_0,bert_svd_1,bert_svd_2,bert_svd_3,bert_svd_4,bert_svd_5,bert_svd_6,bert_svd_7,bert_svd_8,bert_svd_9,...,bert_svd_55,bert_svd_56,bert_svd_57,bert_svd_58,bert_svd_59,bert_svd_60,bert_svd_61,bert_svd_62,bert_svd_63,book_id
0,1.953407,-0.009939,-0.175643,-0.216708,0.574654,-0.015705,0.064294,0.344417,0.090202,-0.230643,...,0.104362,0.007713,-0.134847,0.063232,0.149613,-0.019946,-0.039851,-0.012789,-0.045084,20
1,2.066995,-1.106735,-0.150448,0.036450,-0.041883,0.270460,-0.033279,-0.332856,-0.309212,0.236490,...,0.128414,-0.080793,-0.060765,0.184729,-0.018742,-0.009717,0.217395,0.009724,0.135374,35
2,1.393094,1.287392,-0.506393,-0.117092,-0.608263,0.064698,-0.567067,0.228929,-0.097867,0.050556,...,0.250986,-0.111380,0.037080,-0.040330,0.078609,0.020037,0.071419,-0.164189,0.155958,52
3,1.816526,-0.975894,-0.158566,-0.034420,0.099468,0.408841,0.574383,0.415603,-0.434942,-0.219015,...,-0.089327,0.011864,0.163004,-0.048630,0.014325,0.095398,0.157875,0.022487,0.051967,54
4,1.790848,-0.162382,0.130937,-0.656481,0.406806,0.176745,0.317582,0.455732,-0.103915,0.322708,...,0.122606,-0.014003,-0.080316,-0.024489,0.143148,0.059063,-0.185523,-0.090119,0.000951,69


## 3. Таргет 0/1/2 и генерация "холодных" негативов

In [5]:
# has_read=1 -> 2, has_read=0 -> 1
train_df[COL_REL] = np.where(train_df[COL_HAS_READ] == 1, 2, 1).astype("int8")

def generate_negative_samples(
    train_df: pd.DataFrame,
    books_df: pd.DataFrame,
    negatives_per_user: int = 15,
    random_state: int = 42,
) -> pd.DataFrame:
    """Для каждого пользователя сэмплируем книги, с которыми он никогда не взаимодействовал.

    Эти книги имитируют "холодных" кандидатов (rel=0) из candidates.csv.
    """
    rng = np.random.default_rng(random_state)
    all_book_ids = books_df[COL_BOOK_ID].unique()

    user2books = (
        train_df
        .groupby(COL_USER_ID)[COL_BOOK_ID]
        .apply(set)
        .to_dict()
    )

    rows: list[dict] = []
    for user_id, seen in tqdm(user2books.items(), desc="Negative sampling"):
        seen_arr = np.fromiter(seen, dtype=all_book_ids.dtype)
        candidates = np.setdiff1d(all_book_ids, seen_arr)
        if len(candidates) == 0:
            continue
        sample_size = min(negatives_per_user, len(candidates))
        sampled = rng.choice(candidates, size=sample_size, replace=False)
        for b in sampled:
            rows.append(
                {
                    COL_USER_ID: user_id,
                    COL_BOOK_ID: int(b),
                    COL_REL: 0,
                    COL_HAS_READ: 0,
                }
            )
    neg_df = pd.DataFrame(rows)
    print(f"Сгенерировано негативов: {len(neg_df):,}")
    return neg_df

neg_df = generate_negative_samples(
    train_df=train_df,
    books_df=books_df,
    negatives_per_user=NEGATIVES_PER_USER,
    random_state=RANDOM_STATE,
)

# Добавим timestamp для негативов чуть меньше минимума, чтобы не ломать временной сплит
train_df[COL_TIMESTAMP] = pd.to_datetime(train_df[COL_TIMESTAMP])
if not neg_df.empty:
    neg_df[COL_TIMESTAMP] = train_df[COL_TIMESTAMP].min() - pd.Timedelta(days=1)

full_train_df = pd.concat([train_df, neg_df], ignore_index=True)
print("full_train_df:", full_train_df.shape)
full_train_df.head()


Negative sampling: 100%|██████████| 7289/7289 [00:10<00:00, 717.30it/s]


Сгенерировано негативов: 109,335
full_train_df: (378396, 6)


,user_id,book_id,has_read,rating,timestamp,rel_target
0,3870,310170,0,0.0,2008-04-27 21:06:16,1
1,3870,306406,0,0.0,2008-06-07 11:51:01,1
2,4091,195676,0,0.0,2008-08-06 00:40:55,1
3,3870,554261,1,8.0,2008-08-07 09:16:12,2
4,3870,33078,1,2.0,2008-08-07 09:17:20,2


## 4. Фича-инжиниринг: user / book / жанры / BERT

In [6]:
def build_user_features(df: pd.DataFrame) -> pd.DataFrame:
    """User‑level агрегаты по реальным взаимодействиям (rel > 0)."""
    print("Строим user‑фичи...")
    hist = df[df[COL_REL] > 0].copy()

    agg = hist.groupby(COL_USER_ID).agg(
        user_n_interactions=(COL_REL, "size"),
        user_n_read=(COL_REL, lambda x: (x == 2).sum()),
        user_n_planned=(COL_REL, lambda x: (x == 1).sum()),
    ).reset_index()

    agg["user_read_ratio"] = agg["user_n_read"] / agg["user_n_interactions"]
    agg["user_plan_ratio"] = agg["user_n_planned"] / agg["user_n_interactions"]

    if COL_RATING in hist.columns:
        ragg = hist.groupby(COL_USER_ID).agg(
            user_mean_rating=(COL_RATING, "mean"),
            user_median_rating=(COL_RATING, "median"),
        ).reset_index()
        agg = agg.merge(ragg, on=COL_USER_ID, how="left")

    return agg


def build_book_features(df: pd.DataFrame) -> pd.DataFrame:
    """Book‑level агрегаты и глобальная конверсия план -> чтение."""
    print("Строим book‑фичи...")
    hist = df[df[COL_REL] > 0].copy()

    agg = hist.groupby(COL_BOOK_ID).agg(
        book_n_interactions=(COL_REL, "size"),
        book_n_read=(COL_REL, lambda x: (x == 2).sum()),
        book_n_planned=(COL_REL, lambda x: (x == 1).sum()),
    ).reset_index()

    agg["book_read_ratio"] = agg["book_n_read"] / agg["book_n_interactions"]
    agg["book_plan_ratio"] = agg["book_n_planned"] / agg["book_n_interactions"]
    agg["book_conversion"] = agg["book_n_read"] / (agg["book_n_planned"] + 1e-6)

    if COL_RATING in hist.columns:
        ragg = hist.groupby(COL_BOOK_ID).agg(
            book_mean_rating=(COL_RATING, "mean"),
            book_median_rating=(COL_RATING, "median"),
        ).reset_index()
        agg = agg.merge(ragg, on=COL_BOOK_ID, how="left")

    # Примёрджим некоторые поля из books.csv (год, язык, avg_rating и т.п.)
    cols_from_books = [
        COL_BOOK_ID,
        "author_id",
        "publication_year",
        "language",
        "avg_rating",
        "publisher",
    ]
    existing_cols = [c for c in cols_from_books if c in books_df.columns]
    books_sub = books_df[existing_cols].drop_duplicates(subset=[COL_BOOK_ID])
    agg = agg.merge(books_sub, on=COL_BOOK_ID, how="left")

    return agg


user_features = build_user_features(full_train_df)
book_features = build_book_features(full_train_df)

user_features.head(), book_features.head()


Строим user‑фичи...
Строим book‑фичи...


(   user_id  user_n_interactions  user_n_read  user_n_planned  user_read_ratio  \
 0      151                   75           36              39         0.480000   
 1      210                   31            0              31         0.000000   
 2      560                    6            0               6         0.000000   
 3     1380                   56           29              27         0.517857   
 4     1850                   77           38              39         0.493506   
 
    user_plan_ratio  user_mean_rating  user_median_rating  
 0         0.520000          3.840000                 0.0  
 1         1.000000          0.000000                 0.0  
 2         1.000000          0.000000                 0.0  
 3         0.482143          3.892857                 3.0  
 4         0.506494          2.519481                 0.0  ,
    book_id  book_n_interactions  book_n_read  book_n_planned  book_read_ratio  \
 0       20                  122          103              19  

In [8]:
# Жанры книги
book_genres_map: dict[int, set[int]] = (
    book_genres_df
    .groupby(COL_BOOK_ID)["genre_id"]
    .apply(lambda s: set(s.dropna().astype(int)))
    .to_dict()
)

def build_user_genre_profile(df: pd.DataFrame) -> dict[int, dict[int, int]]:
    """user_id -> {genre_id: count} по реальным взаимодействиям."""
    print("Строим жанровые профили пользователей...")
    hist = df[df[COL_REL] > 0].copy()
    merged = hist[[COL_USER_ID, COL_BOOK_ID, COL_REL]].merge(
        book_genres_df[[COL_BOOK_ID, "genre_id"]],
        on=COL_BOOK_ID,
        how="left",
    )
    merged = merged.dropna(subset=["genre_id"])
    merged["genre_id"] = merged["genre_id"].astype(int)

    user_genre_counts: dict[int, dict[int, int]] = {}
    for (user_id, genre_id), g in merged.groupby([COL_USER_ID, "genre_id"]):
        cnt = len(g)
        user_genre_counts.setdefault(user_id, {})[genre_id] = cnt

    return user_genre_counts


user_genre_profile = build_user_genre_profile(full_train_df)

def calc_user_book_genre_match(user_id: int, book_id: int) -> tuple[float, float]:
    """Возвращает (Jaccard, доля жанров книги, присутствующих у пользователя)."""
    book_genres = book_genres_map.get(book_id, set())
    if not book_genres:
        return 0.0, 0.0

    user_profile = user_genre_profile.get(user_id, {})
    user_genres = set(user_profile.keys())
    if not user_genres:
        return 0.0, 0.0

    inter = len(book_genres & user_genres)
    union = len(book_genres | user_genres)
    jaccard = inter / union if union > 0 else 0.0
    frac_known = inter / len(book_genres)

    return jaccard, frac_known


def add_bert_features(df: pd.DataFrame, bert_df: pd.DataFrame) -> pd.DataFrame:
    return df.merge(bert_df, on=COL_BOOK_ID, how="left")


def add_user_book_aggregate_features(
    df: pd.DataFrame,
    user_feat: pd.DataFrame,
    book_feat: pd.DataFrame,
) -> pd.DataFrame:
    df = df.merge(user_feat, on=COL_USER_ID, how="left")
    df = df.merge(book_feat, on=COL_BOOK_ID, how="left")
    # добавим простейшие фичи из users.csv (пол, возраст)
    if "gender" in users_df.columns or "age" in users_df.columns:
        cols = [COL_USER_ID]
        for c in ["gender", "age"]:
            if c in users_df.columns:
                cols.append(c)
        df = df.merge(users_df[cols].drop_duplicates(subset=[COL_USER_ID]), on=COL_USER_ID, how="left")
    return df


def add_user_book_match_features(df: pd.DataFrame) -> pd.DataFrame:
    print("Добавляем user‑book жанровые matching‑фичи...")
    jaccards: list[float] = []
    frac_knowns: list[float] = []

    for row in tqdm(df[[COL_USER_ID, COL_BOOK_ID]].itertuples(index=False), total=len(df)):
        u_id, b_id = row
        j, fk = calc_user_book_genre_match(int(u_id), int(b_id))
        jaccards.append(j)
        frac_knowns.append(fk)

    df["user_book_genre_jaccard"] = jaccards
    df["user_book_genre_frac_known"] = frac_knowns
    return df


Строим жанровые профили пользователей...


## 5. Временной сплит train → train/val

In [9]:
def temporal_split_by_fraction(
    df: pd.DataFrame,
    ts_col: str,
    train_fraction: float = 0.8,
) -> tuple[np.ndarray, np.ndarray]:
    assert 0 < train_fraction < 1
    cutoff = df[ts_col].quantile(train_fraction)
    train_mask = df[ts_col] <= cutoff
    val_mask = ~train_mask
    return train_mask.values, val_mask.values


full_train_df[COL_TIMESTAMP] = pd.to_datetime(full_train_df[COL_TIMESTAMP])
full_train_df["ts_int"] = full_train_df[COL_TIMESTAMP].view("int64")

train_mask, val_mask = temporal_split_by_fraction(
    full_train_df, "ts_int", train_fraction=0.8
)

train_split = full_train_df[train_mask].copy()
val_split = full_train_df[val_mask].copy()

print("train_split:", train_split.shape)
print("val_split:", val_split.shape)


train_split: (302717, 7)
val_split: (75679, 7)


## 6. Построение фичей для train / val

In [10]:
def build_features_for_interactions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = add_user_book_aggregate_features(out, user_features, book_features)
    out = add_bert_features(out, bert_svd_df)
    out = add_user_book_match_features(out)
    return out

print("Строим фичи для train_split...")
train_features_df = build_features_for_interactions(train_split)

print("Строим фичи для val_split...")
val_features_df = build_features_for_interactions(val_split)

train_features_df.shape, val_features_df.shape


Строим фичи для train_split...
Добавляем user‑book жанровые matching‑фичи...


100%|██████████| 302717/302717 [00:01<00:00, 212560.38it/s]


Строим фичи для val_split...
Добавляем user‑book жанровые matching‑фичи...


100%|██████████| 75679/75679 [00:00<00:00, 201569.35it/s]


((302717, 95), (75679, 95))

## 7. Подготовка данных для LightGBM Ranker

In [11]:
exclude_cols = [
    COL_REL,
    COL_USER_ID,
    COL_BOOK_ID,
    COL_TIMESTAMP,
    "ts_int",
    COL_HAS_READ,
]

features = [c for c in train_features_df.columns if c not in exclude_cols]
print("Число фичей:", len(features))

# сортируем по user_id для групп
train_features_df = train_features_df.sort_values(COL_USER_ID).reset_index(drop=True)
val_features_df = val_features_df.sort_values(COL_USER_ID).reset_index(drop=True)

X_train = train_features_df[features].copy()
y_train = train_features_df[COL_REL].astype("int32").values

X_val = val_features_df[features].copy()
y_val = val_features_df[COL_REL].astype("int32").values

# float64 -> float32
float_cols = X_train.select_dtypes(include=["float64"]).columns
if len(float_cols) > 0:
    X_train[float_cols] = X_train[float_cols].astype("float32")
    X_val[float_cols] = X_val[float_cols].astype("float32")

group_train = (
    train_features_df.groupby(COL_USER_ID)
    .size()
    .to_numpy()
)
group_val = (
    val_features_df.groupby(COL_USER_ID)
    .size()
    .to_numpy()
)

print("Групп train:", len(group_train), "средний размер:", group_train.mean())
print("Групп val:", len(group_val), "средний размер:", group_val.mean())


Число фичей: 89
Групп train: 7289 средний размер: 41.53066264233777
Групп val: 4688 средний размер: 16.143131399317404


## 8. Обучение LightGBM Ranker (LambdaMART)

In [12]:
lgb_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "label_gain": [0, 1, 2],
    "eval_at": [NDCG_K],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 0.1,
    "verbose": -1,
    "n_jobs": -1,
    "seed": RANDOM_STATE,
    "boosting_type": "gbdt",
    "max_bin": 255,
    "force_row_wise": True,
}

print(f"Обучаем LightGBM Ranker на {len(features)} фичах...")

ranker = lgb.LGBMRanker(**lgb_params)

callbacks = [
    lgb.early_stopping(stopping_rounds=100, verbose=True),
    lgb.log_evaluation(period=20),
]

ranker.fit(
    X_train,
    y_train,
    group=group_train,
    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
    eval_at=[NDCG_K],
    callbacks=callbacks,
)

booster = ranker.booster_
booster.save_model(str(MODEL_DIR / MODEL_FILENAME))

with open(MODEL_DIR / "features_list.json", "w") as f:
    json.dump(features, f)

print("Модель сохранена в", MODEL_DIR / MODEL_FILENAME)


Обучаем LightGBM Ranker на 89 фичах...
Training until validation scores don't improve for 100 rounds
[20]	valid_0's ndcg@20: 0.998648
[40]	valid_0's ndcg@20: 0.998732
[60]	valid_0's ndcg@20: 0.998911
[80]	valid_0's ndcg@20: 0.998992
[100]	valid_0's ndcg@20: 0.998987
Did not meet early stopping. Best iteration is:
[91]	valid_0's ndcg@20: 0.999024
Модель сохранена в models/lgbm_ranker_bert.txt


## 9. Локальная метрика NDCG@20

In [13]:
def dcg_at_k(rels: list[int], k: int) -> float:
    rels_arr = np.asfarray(rels)[:k]
    if rels_arr.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rels_arr.size + 2))
    return float(np.sum(rels_arr / discounts))

def ndcg_at_k(rels: list[int], k: int) -> float:
    actual_dcg = dcg_at_k(rels, k)
    ideal_rels = sorted(rels, reverse=True)
    ideal_dcg = dcg_at_k(ideal_rels, k)
    if ideal_dcg == 0:
        return 0.0
    return float(actual_dcg / ideal_dcg)

val_scores = ranker.predict(X_val)

val_eval_df = val_features_df[[COL_USER_ID, COL_BOOK_ID, COL_REL]].copy()
val_eval_df["pred_score"] = val_scores

user_ndcgs: list[float] = []
for user_id, g in val_eval_df.groupby(COL_USER_ID):
    g_sorted = g.sort_values("pred_score", ascending=False)
    rels = g_sorted[COL_REL].tolist()
    ndcg = ndcg_at_k(rels, NDCG_K)
    user_ndcgs.append(ndcg)

local_ndcg = float(np.mean(user_ndcgs))
print(f"Local NDCG@{NDCG_K}: {local_ndcg:.5f}")


Local NDCG@20: 0.99902


## 10. Предсказания для candidates.csv и формирование submission

In [14]:
def explode_candidates(cand_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict] = []
    for row in tqdm(cand_df.itertuples(index=False), total=len(cand_df), desc="Explode candidates"):
        user_id = getattr(row, COL_USER_ID)
        book_list_str = getattr(row, "book_id_list")
        book_ids = [x.strip() for x in str(book_list_str).split(",") if x.strip() != ""]
        for b in book_ids:
            rows.append({COL_USER_ID: int(user_id), COL_BOOK_ID: int(b)})
    return pd.DataFrame(rows)


candidates_long = explode_candidates(candidates_df)
print("candidates_long:", candidates_long.shape)
candidates_long.head()


Explode candidates: 100%|██████████| 3512/3512 [00:00<00:00, 97948.62it/s]

candidates_long: (81048, 2)


,user_id,book_id
0,210,11936
1,210,254097
2,210,709075
3,210,840500
4,210,971259


In [15]:
def build_features_for_candidates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # timestamp/has_read в тесте неизвестны → ставим фиктивные значения
    out[COL_HAS_READ] = 0
    out[COL_TIMESTAMP] = full_train_df[COL_TIMESTAMP].max() + pd.Timedelta(days=1)
    out["ts_int"] = out[COL_TIMESTAMP].view("int64")

    out = add_user_book_aggregate_features(out, user_features, book_features)
    out = add_bert_features(out, bert_svd_df)
    out = add_user_book_match_features(out)
    return out

print("Строим фичи для candidates_long...")
cand_features_df = build_features_for_candidates(candidates_long)
cand_features_df.shape


Строим фичи для candidates_long...
Добавляем user‑book жанровые matching‑фичи...


100%|██████████| 81048/81048 [00:00<00:00, 287514.79it/s]


(81048, 93)

In [22]:
# Гарантируем, что в cand_features_df есть все колонки из features
missing_cols = [c for c in features if c not in cand_features_df.columns]
print("Missing feature columns in candidates:", missing_cols)

for c in missing_cols:
    # логично для rating — 0, для остальных тоже можно 0
    cand_features_df[c] = 0.0

X_cand = cand_features_df[features].copy()
float_cols = X_cand.select_dtypes(include=["float64"]).columns
if len(float_cols) > 0:
    X_cand[float_cols] = X_cand[float_cols].astype("float32")

cand_scores = ranker.predict(X_cand)
cand_features_df["pred_score"] = cand_scores

# 1) Удаляем возможные дубликаты (user_id, book_id)
cand_features_df = cand_features_df.drop_duplicates(
    subset=[COL_USER_ID, COL_BOOK_ID]
)

# 2) Сортируем по user_id и предсказанному скору
cand_sorted = cand_features_df.sort_values(
    [COL_USER_ID, "pred_score"],
    ascending=[True, False]
)

# 3) Берём топ-20 книг на каждого пользователя
cand_topk = (
    cand_sorted
    .groupby(COL_USER_ID, as_index=False)
    .head(20)   # гарантирует len <= 20
)

# 4) Собираем submission в нужный формат
submission = (
    cand_topk
    .groupby(COL_USER_ID)[COL_BOOK_ID]
    .apply(lambda ids: ",".join(map(str, ids.tolist())))
    .reset_index()
    .rename(columns={COL_BOOK_ID: "book_id_list"})
)

submission_path = SUBMISSION_DIR / SUBMISSION_FILENAME
submission.to_csv(submission_path, index=False)
print("Submission сохранён в", submission_path)

submission.head()



Missing feature columns in candidates: []
Submission сохранён в submissions/submission_bert_ltr.csv


,user_id,book_id_list
0,210,"2274394,11936,2370751,2600001,2195786,3015694,..."
1,1380,"2290484,2548861,2231328,2356900,998313,8369,28..."
2,2050,"1021078,460492,2300795,2347744,308364,1918727,..."
3,2740,"987516,112023,549194,2307893,5535190,1834192,2..."
4,4621,"2274394,2225251,2458413,2446687,2191492,234756..."


In [23]:
# %% [markdown]
# ## Валидация submission: уникальность, длина, соответствие candidates

# %% [code]
def validate_submission(submission_path, candidates_path, targets_path):
    print("🔍 Проверка корректности submission...")
    
    submission = pd.read_csv(submission_path)
    candidates = pd.read_csv(candidates_path)
    targets = pd.read_csv(targets_path)
    
    # Создаём словарь: user_id → множество разрешённых book_id
    candidates_dict = {}
    for _, row in candidates.iterrows():
        user_id = row['user_id']
        book_ids = set(map(int, row['book_id_list'].split(',')))
        candidates_dict[user_id] = book_ids

    # Проверка 1: все user_id из targets есть в submission
    submission_users = set(submission['user_id'])
    targets_users = set(targets['user_id'])
    if submission_users != targets_users:
        missing = targets_users - submission_users
        extra = submission_users - targets_users
        if missing:
            print(f"❌ Отсутствуют пользователи в submission: {sorted(missing)[:5]}...")
        if extra:
            print(f"❌ Лишние пользователи в submission: {sorted(extra)[:5]}...")
        raise AssertionError("Несоответствие списка пользователей!")

    # Проверка 2: для каждого пользователя
    for _, row in submission.iterrows():
        user_id = row['user_id']
        pred_books_str = row['book_id_list']
        
        if not isinstance(pred_books_str, str):
            raise AssertionError(f"Некорректный формат book_id_list для user_id={user_id}")
        
        pred_books = list(map(int, pred_books_str.split(',')))
        pred_set = set(pred_books)
        allowed_books = candidates_dict[user_id]

        # Уникальность
        if len(pred_books) != len(pred_set):
            dupes = [bid for bid in pred_books if pred_books.count(bid) > 1]
            raise AssertionError(f"Дубликаты у user_id={user_id}: {dupes[:3]}")

        # Длина ≤ 20
        if len(pred_books) > 20:
            raise AssertionError(f"Больше 20 книг у user_id={user_id}: {len(pred_books)}")

        # Все книги — из candidates
        if not pred_set.issubset(allowed_books):
            invalid = pred_set - allowed_books
            raise AssertionError(f"Запрещённые книги у user_id={user_id}: {sorted(invalid)[:3]}")

    print("✅ Submission прошёл все проверки!")

# Запуск проверки
SUBMISSION_PATH = f"/kaggle/working/submissions/submission_bert_ltr.csv"
CANDIDATES_PATH = f"/kaggle/input/nto-2-tour-team/candidates.csv"
TARGETS_PATH = f"/kaggle/input/nto-2-tour-team/targets.csv"

validate_submission(SUBMISSION_PATH, CANDIDATES_PATH, TARGETS_PATH)

🔍 Проверка корректности submission...
✅ Submission прошёл все проверки!
